# KneeAxis — Step 3: 3D U-Net Knee Segmentation Training

Trains the TensorFlow 3D U-Net model on the **OAIZIB-CM** dataset (Zenodo Record 14934086).

### Anatomical Scope:
Predicts **6 distinct classes**:
- `0`: Background
- `1`: Femur (Distal Bone)
- `2`: Femoral Cartilage
- `3`: Tibia (Proximal Bone)
- `4`: Medial Tibial Cartilage
- `5`: Lateral Tibial Cartilage

> **Anatomical Boundary Note:**
> OAIZIB-CM does not include patella or fibula annotations. For pre-annotation or whole-knee CT baselines, see TotalSegmentator in `guides/DATASET_DOWNLOADS.md`.

### 1. Initialize GPU & Configure Session

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.common.gpu_utils import configure_gpu_memory_growth
from src.segmentation.train import train_segmentation

configure_gpu_memory_growth()

### 2. Set Session Parameters
Configure maximum training minutes and optional resumption ID.

In [ ]:
MAX_MINUTES = 120.0     # Maximum training time (e.g. 2 hours)
RESUME_RUN_ID = None    # Set to existing run ID string to resume

print(f"Target Session Limit: {MAX_MINUTES} minutes")
print(f"Resume Run ID:        {RESUME_RUN_ID or 'Fresh New Run'}")

### 3. Launch 3D U-Net Training
Trains on foreground-aware 3D patches with atomic checkpoint saves after every epoch.

In [ ]:
results = train_segmentation(
    resume_run_id=RESUME_RUN_ID,
    max_minutes=MAX_MINUTES
)
print(f"Segmentation training session complete. Run ID: {results['run_id']}")

### 4. Plot Loss and Mean Foreground Dice Curves

In [ ]:
import json
import matplotlib.pyplot as plt

history_file = Path(results['checkpoint_dir']) / 'training_history.json'
if history_file.exists():
    with open(history_file, 'r') as f:
        hist = json.load(f)
    
    epochs = range(1, len(hist.get('loss', [])) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(epochs, hist.get('loss', []), 'b-', label='Train Loss (Dice+CE)')
    ax1.plot(epochs, hist.get('val_loss', []), 'r--', label='Val Loss')
    ax1.set_title('3D U-Net Total Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(epochs, hist.get('mean_foreground_dice', []), 'g-', label='Train Mean Foreground Dice')
    ax2.plot(epochs, hist.get('val_mean_foreground_dice', []), 'm--', label='Val Mean Foreground Dice')
    ax2.set_title('Mean Foreground Dice Score')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Dice (0-1)')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("History file not found.")